# GR00T N1.7 policy server trên Colab cho `can_nav_pnp`, mô phỏng MuJoCo trên máy local (Ubuntu)

Notebook chạy checkpoint GR00T N1.7 đã finetune bằng `colab_finetune_gr00t_sonic11.ipynb` (embodiment `UNITREE_G1_SONIC`) trên
GPU Colab. Server là `gr00t/eval/run_gr00t_server.py` chuẩn của Isaac-GR00T (ZMQ, endpoint `get_action`). Notebook mở đường hầm
TCP Pinggy để máy local gọi tới. Mô phỏng MuJoCo (cảnh `can_nav_pnp`, SONIC 1.1) chạy trên máy local bằng `closed_loop_eval.py --gr00t`.

Model nhận ảnh `ego_view` 320×240, state 46 chiều (khớp 0810 + trọng lực chiếu) và câu lệnh. Model trả về chunk 40 bước:
`motion_token` 64, `left_hand_joints` 7, `right_hand_joints` 7. SONIC 1.1 ở máy local chạy token này giống như với Ψ₀.

### Yêu cầu
- GPU **L4 hoặc A100**.
- **HF_TOKEN** trong Colab Secrets, với quyền tải `nvidia/Cosmos-Reason2-2B` (giống notebook finetune).
- Checkpoint trên Drive: `My Drive/gr00t/checkpoints/<EXPERIMENT>/checkpoint-N` (bước 8a của notebook finetune chép về đó).

### Các bước
1. Runtime → Change runtime type → **L4** hoặc **A100**.
2. Chạy Bước 1 → 5. Bước 4 chọn checkpoint (mặc định checkpoint mới nhất) và chép về ổ local của Colab. Bước 5 chạy server nền,
   rồi gọi thử một lần đúng định dạng client.
3. Bước 6 mở tunnel và in lệnh cho máy local. Tunnel miễn phí hết hạn sau khoảng 60 phút: chạy lại Bước 6, không cần nạp lại model.
4. Bước 7 xem trạng thái server; Bước 8 dừng server (phải đặt `STOP_SERVER = True`).
5. **Đánh giá ngay trên Colab (khuyên dùng):** Bước 5b cài môi trường mô phỏng (cần `My Drive/can_nav/can_nav_eval.zip`, khoảng 200 MB), Bước 5c chạy vòng kín qua `localhost`: không tunnel, không giật; bằng chứng từng cảnh (video, `overview.png`, `result.json`, `trace.npz`) và `summary.md` chép về `My Drive/gr00t/eval/`.

### Trên máy local (Ubuntu)
Cài thư viện ZMQ cho môi trường mô phỏng (một lần):
```bash
cd ~/Code/PSI/psi0
uv pip install --python .venv-datagen/bin/python pyzmq msgpack
```
Rồi dán lệnh mà Bước 6 in ra. Lệnh có dạng:
```bash
SONIC_VARIANT=sonic_v1_1 .venv-datagen/bin/python scripts/datagen/can_nav_pnp/closed_loop_eval.py \
    --gr00t HOST:PORT --obstacle center --layout-from .data/can_nav_160 \
    --episodes 10 --viewer --video --out .data/eval/gr00t_ckptN
```
- `--obstacle center --layout-from .data/can_nav_160`: cảnh giống dữ liệu train của checkpoint 160 episode. Checkpoint train trên
  bộ 500 episode thì bỏ `--obstacle center`.
- Lần gọi đầu có thể mất vài chục giây (warm-up). Nếu báo timeout thì thêm `--timeout 300`.
- Máy không có màn hình (SSH): bỏ `--viewer`, giữ `--video`.

Isaac-GR00T được ghim ở commit `51d4c89`, giống notebook finetune.

In [ ]:
# [BƯỚC 1] Kiểm tra GPU
!nvidia-smi

In [ ]:
%%bash
# [BƯỚC 2] Cài Isaac-GR00T vào môi trường riêng (.venv, Python 3.12) bằng uv (~5-10 phút).
# set -e: dừng ngay ở lệnh lỗi đầu tiên để thấy rõ nguyên nhân (lệnh "!" của Colab không tự dừng khi lỗi).
set -euxo pipefail
cd /content
[ -d Isaac-GR00T/.git ] || GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/NVIDIA/Isaac-GR00T
cd Isaac-GR00T
git checkout 51d4c89f72fda44cbf77285c6a8114b52676b8a1
# Clone bỏ qua file Git LFS (video, dataset demo) cho nhẹ, nhưng uv cần wheel LFS trong
# scripts/deployment/dgpu/wheels để giải lock (kể cả wheel aarch64 không dùng trên Colab): tải riêng thư mục này.
command -v git-lfs || (apt-get -qq update && apt-get -qq install -y git-lfs)
git lfs install --local
git lfs pull --include="scripts/deployment/dgpu/wheels/*"
# Trình cài uv chọn /usr/local/bin hoặc ~/.local/bin tùy môi trường: gọi uv qua PATH, không dùng đường dẫn cứng
export PATH="$HOME/.local/bin:/usr/local/bin:$PATH"
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
# Venv luôn nằm ở /content/Isaac-GR00T/.venv (bỏ qua biến môi trường nếu có)
unset UV_PROJECT_ENVIRONMENT VIRTUAL_ENV
uv sync --python 3.12
.venv/bin/python -c "import gr00t; print('GR00T installed successfully')"

In [ ]:
# [BƯỚC 3] Nạp HuggingFace token từ Colab Secrets (không dán token trực tiếp vào notebook)
# và kiểm tra ngay quyền tải backbone gated nvidia/Cosmos-Reason2-2B.
# Chạy lại cell này mỗi khi runtime khởi động lại.
import os
from huggingface_hub import hf_hub_download, whoami
from huggingface_hub.utils import GatedRepoError

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"Không đọc được secret HF_TOKEN ({type(e).__name__}); nhập token thủ công.")
    from getpass import getpass
    token = getpass("HuggingFace token: ")
token = (token or "").strip()
assert token, "HF_TOKEN rỗng: thêm secret tên HF_TOKEN (biểu tượng 🔑) và bật Notebook access."
os.environ["HF_TOKEN"] = token

try:
    user = whoami(token=token)
except Exception as e:
    raise RuntimeError(
        "Token không hợp lệ hoặc đã bị thu hồi. Tạo token mới tại https://huggingface.co/settings/tokens "
        "rồi cập nhật secret HF_TOKEN."
    ) from e
print(f"Token hợp lệ, tài khoản: {user['name']}")

try:
    hf_hub_download("nvidia/Cosmos-Reason2-2B", "config.json", token=token)
except GatedRepoError as e:
    raise RuntimeError(
        f"Tài khoản {user['name']} chưa tải được nvidia/Cosmos-Reason2-2B. Kiểm tra:\n"
        "  1. Đã bấm xin quyền và được duyệt tại https://huggingface.co/nvidia/Cosmos-Reason2-2B "
        "(đăng nhập đúng tài khoản trên).\n"
        "  2. Nếu là token fine-grained: bật quyền 'Read access to contents of all public gated repos "
        "you can access', hoặc dùng token loại Read."
    ) from e
print("✅ Có quyền tải nvidia/Cosmos-Reason2-2B.")

In [ ]:
# [BƯỚC 4] Chọn checkpoint trên Drive và chép về ổ local của Colab (nạp thẳng từ Drive rất chậm).
# Chạy lại cell này thì các file đã chép đủ được bỏ qua.
EXPERIMENT = "can_nav_160_gr00t"   # = EXPERIMENT của colab_finetune_gr00t_sonic11.ipynb (<DATASET_NAME>_gr00t)
CKPT_STEP = ""                     # "": checkpoint mới nhất trên Drive; hoặc số bước, ví dụ "8000"
EMBODIMENT_TAG = "UNITREE_G1_SONIC"
PORT = 5555

import os, re, glob, shutil, time
from google.colab import drive
drive.mount("/content/drive")
ckpt_root = f"/content/drive/MyDrive/gr00t/checkpoints/{EXPERIMENT}"
steps = sorted(int(re.search(r"checkpoint-(\d+)$", p).group(1)) for p in glob.glob(f"{ckpt_root}/checkpoint-*")
               if re.search(r"checkpoint-(\d+)$", p))
assert steps, f"Không có checkpoint nào trong {ckpt_root} (bước 8a của notebook finetune chép về đây)."
step = int(CKPT_STEP) if CKPT_STEP else steps[-1]
assert step in steps, f"Không có checkpoint-{step}; trên Drive có: {steps}"
src, MODEL_PATH = f"{ckpt_root}/checkpoint-{step}", f"/content/ckpt/{EXPERIMENT}/checkpoint-{step}"
t0, copied = time.time(), 0
for root, _, files in os.walk(src):
    for f in files:
        s = os.path.join(root, f); d = os.path.join(MODEL_PATH, os.path.relpath(s, src))
        if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s):
            continue
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copyfile(s, d + ".part"); os.replace(d + ".part", d)
        copied += os.path.getsize(d)
        print(f"  {os.path.relpath(d, MODEL_PATH)} ({os.path.getsize(d) / 2**30:.2f} GB)", flush=True)
print(f"checkpoint-{step} (trên Drive có {steps}) -> {MODEL_PATH}, chép {copied / 2**30:.1f} GB trong {time.time() - t0:.0f} s")

In [ ]:
# [BƯỚC 5] Chạy server nền (log: /content/server.log), chờ sẵn sàng, rồi gọi thử get_action đúng định dạng của
# closed_loop_eval.py --gr00t. Bấm ■ chỉ dừng việc chờ, server vẫn chạy.
import os, socket, subprocess, time
assert os.path.exists("/content/Isaac-GR00T/.venv/bin/python"), "Chưa cài Isaac-GR00T: chạy Bước 2."
assert os.environ.get("HF_TOKEN"), "Chưa có HF_TOKEN: chạy Bước 3."
SERVER_LOG = "/content/server.log"

def listening():
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", PORT)) == 0

if listening():
    print(f"Đã có server ở port {PORT} (muốn đổi checkpoint thì dừng ở Bước 8 trước).")
else:
    server = subprocess.Popen(
        ["/content/Isaac-GR00T/.venv/bin/python", "gr00t/eval/run_gr00t_server.py", "--model-path", MODEL_PATH,
         "--embodiment-tag", EMBODIMENT_TAG, "--host", "0.0.0.0", "--port", str(PORT)],
        cwd="/content/Isaac-GR00T", stdout=open(SERVER_LOG, "w"), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        env={k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "MPLBACKEND")}, start_new_session=True)
    print(f"Đang nạp model (PID {server.pid}), log {SERVER_LOG} ...")
    t0 = time.time()
    while not listening():
        if server.poll() is not None:
            print(open(SERVER_LOG, errors="replace").read()[-4000:])
            raise RuntimeError(f"Server thoát với mã {server.returncode}, xem log ở trên.")
        time.sleep(5)
    print(f"Server sẵn sàng sau {time.time() - t0:.0f} s")

# gọi thử bằng PolicyClient của Isaac-GR00T, cùng cấu trúc quan sát mà closed_loop_eval.py gửi
TEST = r"""
import time, numpy as np
from gr00t.policy.server_client import PolicyClient
keys = [("left_leg", 6), ("right_leg", 6), ("waist", 3), ("left_arm", 7), ("right_arm", 7), ("left_hand", 7),
        ("right_hand", 7), ("projected_gravity", 3)]
obs = {"video": {"ego_view": np.zeros((1, 1, 240, 320, 3), np.uint8)},
       "state": {k: np.zeros((1, 1, n), np.float32) for k, n in keys},
       "language": {"annotation.human.task_description": [["warm up"]]}}
c = PolicyClient(host="127.0.0.1", port=%d, timeout_ms=300000)
for i in range(2):
    t = time.time(); action, _ = c.get_action(obs)
    print(f"gọi thử {i + 1}: " + ", ".join(f"{k} {tuple(v.shape)}" for k, v in action.items()) + f" trong {time.time() - t:.2f} s")
""" % PORT
r = subprocess.run(["/content/Isaac-GR00T/.venv/bin/python", "-c", TEST], cwd="/content/Isaac-GR00T",
                   capture_output=True, text=True)
print(r.stdout[-2000:] or r.stderr[-3000:])

In [ ]:
%%bash -s "/content/drive/MyDrive/can_nav"
# [BƯỚC 5b] Môi trường mô phỏng trên Colab (cho đánh giá vòng kín ở Bước 5c); cần Drive đã mount (Bước 4)
# Mô phỏng MuJoCo + SONIC 1.1 chạy ngay trên máy Colab này (Python 3.11 + đúng phiên bản thư viện như máy local), gọi server
# qua localhost: không cần tunnel, mỗi request chỉ tốn thời gian suy luận. Cần gói can_nav_eval.zip (~200 MB, pack_colab.py
# --eval; ~/Downloads/can_nav_upload/drive_can_nav/) trong My Drive/can_nav/; gói can_nav_datagen.zip (~850 MB) cũng
# dùng được. Lần đầu khoảng 2-3 phút.
set -euo pipefail
SRC="$1/can_nav_eval.zip"
[ -f "$SRC" ] || SRC="$1/can_nav_datagen.zip"
[ -f "$SRC" ] || [ -d /content/can_nav/psi0 ] || { echo "Không thấy $1/can_nav_eval.zip: tải ~/Downloads/can_nav_upload/drive_can_nav/can_nav_eval.zip lên đó."; exit 1; }
cd /content
if [ ! -d can_nav/psi0 ]; then
    [ -f /content/can_nav_pack.zip ] || cp "$SRC" /content/can_nav_pack.zip
    mkdir -p can_nav && unzip -q /content/can_nav_pack.zip -d can_nav
fi
export PATH="$HOME/.local/bin:$PATH"
command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh
[ -x /content/venv-sim/bin/python ] || uv venv -q --python 3.11 /content/venv-sim
uv pip install -q --python /content/venv-sim/bin/python -r can_nav/psi0/requirements_datagen.txt pyzmq msgpack
(dpkg -s libegl1 >/dev/null 2>&1) || (apt-get -qq update && apt-get -qq install -y libegl1 libgl1 > /dev/null)
# EGL phải chạy trên GPU NVIDIA: không có file vendor này, libglvnd rơi về Mesa llvmpipe (render bằng CPU, chậm khoảng
# 40 lần: mỗi request cách nhau khoảng 20 s). Cùng cách với hướng dẫn chạy MuJoCo trên Colab của dm_control.
mkdir -p /usr/share/glvnd/egl_vendor.d
[ -f /usr/share/glvnd/egl_vendor.d/10_nvidia.json ] || \
    echo '{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}' > /usr/share/glvnd/egl_vendor.d/10_nvidia.json
MUJOCO_GL=egl PYOPENGL_PLATFORM=egl /content/venv-sim/bin/python - <<'PY'
import sys, time
sys.path.insert(0, "/content/can_nav/psi0/scripts/datagen/can_nav_pnp")
import mujoco, numpy as np
from OpenGL import GL
from scene import sample_params, build_model
m = build_model(sample_params(1)); d = mujoco.MjData(m); mujoco.mj_forward(m, d)
r = mujoco.Renderer(m, 480, 640); r.update_scene(d, camera="head_camera"); img = r.render()
gpu = GL.glGetString(GL.GL_RENDERER).decode()
t = time.time()
for _ in range(20):
    r.update_scene(d, camera="debug_cam"); r.render()
ms = (time.time() - t) / 20 * 1000
noise = float(np.abs(np.diff(img.astype(np.int16), axis=1)).mean())
r.close()
assert noise < 20, f"EGL render looks broken (pixel-step {noise:.1f})"
print(f"render bằng: {gpu}, {ms:.0f} ms/khung 640x480")
assert "llvmpipe" not in gpu.lower() and "software" not in gpu.lower(), \
    "EGL đang render bằng CPU (llvmpipe): rất chậm. Kiểm tra runtime có GPU, rồi khởi động lại runtime và chạy lại."
print(f"mô phỏng sẵn sàng: mujoco {mujoco.__version__}, render EGL trên GPU OK")
PY

In [ ]:
# [BƯỚC 5c] Đánh giá vòng kín NGAY TRÊN COLAB (không tunnel)
# Mô phỏng gọi server GR00T ở localhost (ZMQ). Mỗi cảnh lưu bằng chứng (video camera đầu + góc ngoài, overview.png,
# result.json, trace.npz) vào /content/eval/<RUN_NAME>/ và được chép về My Drive/gr00t/eval/<RUN_NAME>/ khi cảnh xong.
# Trong lúc chạy, cell hiện ảnh TRỰC TIẾP của cảnh đang chạy; mỗi cảnh xong in kết quả và ảnh tổng hợp; bấm ■ chỉ dừng việc theo dõi, đánh giá vẫn chạy
# (chạy lại cell để theo dõi tiếp). Muốn dừng hẳn: đặt STOP_EVAL = True rồi chạy cell.

EPISODES = 10
SEED_START = 1000000       # các seed đánh giá (>= 1 000 000, không có trong dữ liệu train)
OBSTACLE = "center"        # "center": vật cản ở giữa như can_nav_160; "random": lệch ngẫu nhiên (checkpoint bộ 500 episode)
EXEC_STEPS = 15            # bước action thực thi mỗi chunk trước khi hỏi lại (như closed_loop_eval.py mặc định)
STOP_EVAL = False

import glob, json, os, pathlib, shutil, socket, subprocess, time
from IPython.display import Image as IPImage, Markdown, display
assert os.path.exists("/content/venv-sim/bin/python"), "chạy cell cài môi trường mô phỏng trước"


with socket.socket() as s:
    s.settimeout(1)
    assert s.connect_ex(("127.0.0.1", PORT)) == 0, "server chưa chạy: chạy Bước 5 trước"
RUN_NAME = f"{EXPERIMENT}_ckpt{step}_{OBSTACLE}_colab"
POLICY_ARGS = ["--gr00t", f"127.0.0.1:{PORT}"]
DRIVE_EVAL = pathlib.Path("/content/drive/MyDrive/gr00t/eval") / RUN_NAME
DRIVE_EVAL.mkdir(parents=True, exist_ok=True)
done_on_drive = {p.name for p in DRIVE_EVAL.glob("seed_*")}
DRIVE_TXT = str(DRIVE_EVAL)
DRIVE_LINK = None
for _ in range(10):   # the mount learns the new folder's Drive id after a few seconds
    try:
        DRIVE_LINK = "https://drive.google.com/drive/folders/" + os.getxattr(DRIVE_EVAL, "user.drive.id").decode()
        break
    except OSError:
        time.sleep(3)
print(f"Video và bằng chứng từng cảnh (mở trong trình duyệt): {DRIVE_LINK or 'My Drive/gr00t/eval/' + RUN_NAME}")

def to_drive(path):   # a scene folder (once) or a summary file (each time)
    if path.is_dir():
        shutil.copytree(path, DRIVE_EVAL / path.name, dirs_exist_ok=True)
    else:
        shutil.copy(path, DRIVE_EVAL / path.name)

OUT = pathlib.Path("/content/eval") / RUN_NAME
EVAL_LOG = OUT.parent / f"{RUN_NAME}.log"
if STOP_EVAL:
    subprocess.run(["pkill", "-f", "closed_loop_eval.py"]); print("đã dừng đánh giá")
else:
    if "eval_proc" not in globals() or eval_proc.poll() is not None:
        OUT.parent.mkdir(parents=True, exist_ok=True)
        cmd = ["/content/venv-sim/bin/python", "/content/can_nav/psi0/scripts/datagen/can_nav_pnp/closed_loop_eval.py",
               *POLICY_ARGS, "--episodes", str(EPISODES), "--seed-start", str(SEED_START),
               "--obstacle", OBSTACLE, "--layout-from", "/content/can_nav/psi0/.data/can_nav_160",
               "--exec-steps", str(EXEC_STEPS), "--threads", "4", "--out", str(OUT)]
        env = {**{k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "MPLBACKEND")},
               "SONIC_VARIANT": "sonic_v1_1", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl"}
        eval_proc = subprocess.Popen(cmd, stdout=open(EVAL_LOG, "a"), stderr=subprocess.STDOUT, env=env,
                                     start_new_session=True)
        print("đánh giá bắt đầu:", " ".join(cmd[2:]))

    def sync():
        for d in sorted(OUT.glob("seed_*")):
            if d.is_dir() and not d.name.endswith("_RUNNING") and d.name not in done_on_drive:
                to_drive(d)
                done_on_drive.add(d.name)
                r = json.loads((d / "result.json").read_text())
                print(f"{time.strftime('%H:%M')} {d.name}: policy {r['policy_seconds']} s, grasp {r.get('grasp_at_s')} s, "
                      f"latency {r['latency_mean_s']} s/request", flush=True)
                display(IPImage(filename=str(d / "overview.png"), width=1100))
        for f in ("summary.md", "summary.json", "results.jsonl", "run_config.json"):
            if (OUT / f).exists():
                to_drive(OUT / f)

    # xem trực tiếp: ảnh góc ngoài | camera đầu của cảnh đang chạy (cập nhật mỗi ~0,5 s mô phỏng) + tiến độ
    status_box, live_box = display(Markdown("đang khởi động mô phỏng ..."), display_id=True), display(display_id=True)
    last_live, last_sync = 0.0, 0.0
    while eval_proc.poll() is None:
        f = OUT / "live.jpg"
        if f.exists() and f.stat().st_mtime > last_live:
            last_live = f.stat().st_mtime
            live_box.update(IPImage(filename=str(f), width=960))
            try:
                j = json.loads((OUT / "live.json").read_text())
                n_done = len([d for d in OUT.glob("seed_*") if not d.name.endswith("_RUNNING")])
                status_box.update(Markdown(
                    f"**Đang chạy cảnh {n_done + 1}/{EPISODES}** (seed {j['seed']}): t = {j['t']} s / tối đa 100 s, "
                    f"đã gắp lon: {j['grasped']}, {j['requests']} request ({j['latency_mean_s']} s/request)"))
            except (OSError, ValueError, KeyError):
                pass
        if time.time() - last_sync > 20:
            sync()
            last_sync = time.time()
        time.sleep(2)
    sync()
    status_box.update(Markdown("**Đã chạy xong.**"))
    print(f"xong (mã thoát {eval_proc.returncode}). Bằng chứng: {DRIVE_TXT}\nMở trên Drive: {DRIVE_LINK or DRIVE_TXT}")
    if (OUT / "summary.md").exists():
        display(Markdown((OUT / "summary.md").read_text()))
    else:
        print(open(EVAL_LOG, errors="replace").read()[-3000:])

In [ ]:
# [BƯỚC 6] Mở đường hầm TCP Pinggy qua cổng 443 (không cần tài khoản) và in lệnh cho máy local.
# Tunnel miễn phí hết hạn sau khoảng 60 phút: chạy lại cell này để lấy host/port mới, server không cần khởi động lại.
import subprocess, time
if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    print("Đã đóng tunnel cũ.")
tunnel_proc = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "-p", "443", f"-R0:localhost:{PORT}",
     "tcp@free.pinggy.io"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
pub = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if "tcp://" in line:
        pub = line.strip().split("tcp://")[1].split()[0].rstrip("/")
        break
    if not line:
        time.sleep(0.5)
assert pub and ":" in pub, "Không đọc được địa chỉ tunnel; chạy lại cell này."
print("=" * 90)
print(f"Tunnel tcp://{pub} (mở lúc {time.strftime('%H:%M')} UTC, hết hạn sau khoảng 60 phút)")
print("=" * 90)
print("Trên máy local (terminal Ubuntu), ở thư mục repo psi0. Lần đầu cài ZMQ cho .venv-datagen:")
print("uv pip install --python .venv-datagen/bin/python pyzmq msgpack")
print("Rồi chạy đánh giá vòng kín:")
print(f"SONIC_VARIANT=sonic_v1_1 .venv-datagen/bin/python scripts/datagen/can_nav_pnp/closed_loop_eval.py \\\n"
      f"    --gr00t {pub} --obstacle center --layout-from .data/can_nav_160 \\\n"
      f"    --episodes 10 --viewer --video --out .data/eval/{EXPERIMENT}_ckpt{step}")
print("(checkpoint train trên bộ 500 episode: bỏ --obstacle center)")
print("=" * 90)

In [ ]:
# [BƯỚC 7] Trạng thái server: chạy lại bất cứ lúc nào.
import os, re, socket
log = open("/content/server.log", errors="replace").read() if os.path.exists("/content/server.log") else ""
with socket.socket() as s:
    s.settimeout(1)
    up = s.connect_ex(("127.0.0.1", PORT)) == 0
print("Server:", f"đang lắng nghe ở port {PORT}" if up else "KHÔNG chạy")
print("Tunnel:", "đang mở" if "tunnel_proc" in globals() and tunnel_proc.poll() is None else "đã đóng (chạy Bước 6)")
print(f"Lỗi (Traceback) trong log: {log.count('Traceback')}")
print("--- cuối log ---")
print("\n".join(re.split(r"[\r\n]+", log.strip())[-15:]))

In [ ]:
# [BƯỚC 8] Dừng server (ví dụ để đổi checkpoint ở Bước 4). Tunnel giữ nguyên.
# Phải đặt STOP_SERVER = True mới dừng, để "Chạy tất cả" không tắt server vừa khởi động ở Bước 5.
STOP_SERVER = False
import subprocess, time
if not STOP_SERVER:
    print("Không dừng server. Muốn dừng thì đặt STOP_SERVER = True rồi chạy lại cell này.")
else:
    subprocess.run(["pkill", "-f", "run_gr00t_server.py"])
    time.sleep(3)
    print("Đã dừng server." if subprocess.run(["pgrep", "-f", "run_gr00t_server.py"], capture_output=True).returncode
          else "Server vẫn còn chạy; chạy lại cell này.")